In [ ]:
print(sys.path)

from analysis_village.cc1pi.var_configs import *

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV_grid_new_cc1pi.df", keys2load, 100)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

In [ ]:
mc_evt_df = perform_truth_matching(mc_bnb_evt_df, mc_bnb_nu_df)

In [ ]:
#Load GiBUU dataframe
pot_weight_col = ('slc', 'wgt', '', '', '', '')

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
#mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_GIBUU_gen1.df", keys2load, 100, filter_df = False)
#mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/consolidated_cc1pi_5e18_CV.df", keys2load, 100, filter_df = False)
mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana/cc1pi_5e18_CV.df", keys2load, 100, filter_df = False)

mc_GiBUU_evt_df = mc_GiBUU_df['cc1pi']
mc_GiBUU_nu_df = mc_GiBUU_df['nudf']
mc_GiBUU_hdr_df = mc_GiBUU_df['hdr']

#Add weight column
mc_GiBUU_tot_pot = mc_GiBUU_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_GiBUU_tot_pot))
mc_pot_scale = data_tot_pot / mc_GiBUU_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_GiBUU_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_GiBUU_evt_df))

#Do truth matchign
mc_GiBUU_evt_df = perform_truth_matching(mc_GiBUU_evt_df, mc_GiBUU_nu_df)

# Test background composition

In [ ]:
 mc_evt_df[('slc', 'cut', 'proton_BDT_sideband', '', '', '')] = proton_BDT_sideband_mask(mc_evt_df, ['__ntuple', 'entry', 'rec.slc..index'])

In [ ]:
def add_bad_tracks_column(df):
    track_df = df[df.pfp.trk.len < 3]
    track_counts = track_df.groupby(level=group_levels).size()

    target_key = ('slc','cut_var','n_bad_trks','','','')
    
    # Expand counts to full MultiIndex (broadcast to pfp level)
    df.loc[:,target_key] = df.index.droplevel('rec.slc.reco.pfp..index').map(track_counts)
    
    # Replace NaN (slices with 0 tracks) with 0
    df.loc[:,target_key] = df[target_key].fillna(0)
    
    return df

In [ ]:

def is_MIP_candidate_mask(df):
    is_primary_mask = (df.pfp.parent_is_primary == True) & (df.pfp.dist_to_vertex < CTE.max_primary_distance_to_vertex)
    is_track_mask = (df.pfp.trk.len > CTE.min_track_lenght) & (df.pfp.trackScore > CTE.min_track_score)
    track_mask = is_primary_mask & is_track_mask

    len_mask = df.pfp.trk.len > CTE.MIP_candidate_min_TL
    chi2_mask = (df.pfp.trk.chi2pid.best.chi2_muon < CTE.MIP_candidate_max_muon_score) & (df.pfp.trk.chi2pid.best.chi2_proton > CTE.MIP_candidate_min_proton_score)
    return track_mask & chi2_mask & len_mask

def chi2_cut_mask(df, group_levels):
    chi2_df = df[is_MIP_candidate_mask(df)]
    
    # Count how many pfps per slice
    MIP_counts = chi2_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = MIP_counts[MIP_counts > 2].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask

In [ ]:
def max_angle_cut_mask(df):
    # Returns True if the slice has at least two candidates AND the widest angle is within limits
    # Slices with NaN (0 or 1 candidates) will return False
    return df.slc.measure_var.max_angle_between_candidates < CTE.max_angle_between_candidates

In [ ]:
def proton_BDT_sideband_mask(df, group_levels):
    BDT_proton_df = df[(is_MIP_candidate_mask(df)) & (df.pfp.trk.bdt_proton_score < -2)]
    
    # Count how many pfps per slice
    candidate_counts = BDT_proton_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = candidate_counts[candidate_counts == 1].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask
    

In [ ]:
def shower_cut_mask(df, group_levels):
    is_pandora_primary_mask = (df.pfp.parent_is_primary == True)
    is_shower_mask = (df.pfp.trackScore >= 0) & (df.pfp.trackScore < CTE.max_shower_track_score)
    energy_mask = (df.pfp.shw.bestplane_energy > 0.200) &  (df.pfp.shw.bestplane_energy < 0.500)
    shower_df = df[is_pandora_primary_mask & is_shower_mask & energy_mask] 
        
    # Count how many pfps per slice
    shower_counts = shower_df.groupby(level=group_levels).size()
    
    # Get only slices with at least 2 pfps
    non_valid_slices = shower_counts[(shower_counts > 0)].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(~df.index.droplevel('rec.slc.reco.pfp..index').isin(non_valid_slices), index=df.index)
   
    return final_mask

In [ ]:
group_levels = ['__ntuple', 'entry', 'rec.slc..index']
mc_evt_df = add_bad_tracks_column(mc_evt_df)

In [ ]:
def build_event_cumulative_masks(evt_df, sideband = "shower"):
    # Define which BDT mask to use
    
    cut_sequence = [
        ("cosmic",      evt_df.slc.cut.obvious_cosmic),
        ("t0",          evt_df.slc.cut.t0),
        ("FV",          evt_df.slc.cut.inside_FV),
        ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
        ("track",       evt_df.slc.cut.track),
        ("chi2",        evt_df.slc.cut.MIP_candidates),
        ("shower",      evt_df.slc.cut.shower),
        ("angle",       evt_df.slc.cut.angle),
        ("proton_BDT",  evt_df.slc.cut.proton_BDT),
        ("containment", evt_df.slc.cut.containment),
        ("michel",      evt_df.slc.cut.michel),
        ("extra_pion",  evt_df.slc.cut.extra_pion),
        #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
        ("energy",      evt_df.slc.cut.energy),
    ]
    
    if sideband == "shower": 
        cut_sequence = [
            ("cosmic",      evt_df.slc.cut.obvious_cosmic),
            ("t0",          evt_df.slc.cut.t0),
            ("FV",          evt_df.slc.cut.inside_FV),
            ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
            ("track",       evt_df.slc.cut.track),
            ("chi2",        evt_df.slc.cut.MIP_candidates),
            ("shower",      shower_cut_mask(evt_df, group_levels) == False),
            ("angle",       evt_df.slc.cut.angle),
            #("proton_BDT",  evt_df.slc.cut.proton_BDT),
            ("containment", evt_df.slc.cut.containment),
            ("michel",      evt_df.slc.cut.michel),
            #("extra_pion",  evt_df.slc.cut.extra_pion),
            #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
            #("energy",      evt_df.slc.cut.energy),
            ("energy",      evt_df.slc.cut.obvious_cosmic),
        ]

    if sideband == "two_pions": 
        cut_sequence = [
            ("cosmic",      evt_df.slc.cut.obvious_cosmic),
            ("t0",          evt_df.slc.cut.t0),
            ("FV",          evt_df.slc.cut.inside_FV),
            ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
            ("track",       evt_df.slc.cut.track),
            ("chi2",        evt_df.slc.cut_var.n_MIP_candidates > 2),
            ("shower",      evt_df.slc.cut.shower),
            #("angle",       evt_df.slc.cut.angle),
            #("proton_BDT",  evt_df.slc.cut.proton_BDT),
            ("containment", evt_df.slc.cut.containment),
            ("michel",      evt_df.slc.cut.michel),
            #("extra_pion",  evt_df.slc.cut.extra_pion),
            #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
            ("energy",      evt_df.slc.cut.obvious_cosmic),
        ]
        
    elif sideband == "proton": 
        cut_sequence = [
            ("cosmic",      evt_df.slc.cut.obvious_cosmic),
            ("t0",          evt_df.slc.cut.t0),
            ("FV",          evt_df.slc.cut.inside_FV),
            ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
            ("track",       evt_df.slc.cut.track),
            ("chi2",        evt_df.slc.cut.MIP_candidates),
            ("shower",      evt_df.slc.cut.shower),
            ("angle",       evt_df.slc.cut.angle),
            ("proton_BDT",  proton_BDT_sideband_mask(evt_df,group_levels)),
            ("containment", evt_df.slc.cut.containment),
            ("michel",      evt_df.slc.cut.michel),
            #("extra_pion",  evt_df.slc.cut.extra_pion),
            #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
            #("energy",      evt_df.slc.cut.energy),
            ("energy",      evt_df.slc.cut.obvious_cosmic),
        ]       
        
    
    # 2️⃣ Build cumulative masks
    cumulative_masks = {}
    current_mask = None

    for name, mask in cut_sequence:

        if current_mask is None:
            current_mask = mask.copy()
        else:
            current_mask = current_mask & mask

        cumulative_masks[name] = current_mask.copy()

    return cumulative_masks

In [ ]:
cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
sideband_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "shower")
sideband_proton_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "proton")

sideband_two_pions_cumulative_masks = build_event_cumulative_masks(mc_GiBUU_evt_df, sideband = "two_pions")
GiBUU_cumulative_masks = build_event_cumulative_masks(mc_GiBUU_evt_df, sideband = "")


In [ ]:
mc_signal_df = mc_evt_df[cumulative_masks["energy"]]
mc_control_evt_df = mc_GiBUU_evt_df[sideband_two_pions_cumulative_masks["energy"]]
mc_GiBUU_evt_df = mc_GiBUU_evt_df[GiBUU_cumulative_masks["energy"]]

'''
mc_control_evt_df = mc_evt_df[sideband_cumulative_masks["energy"]]
mc_control_proton_evt_df = mc_evt_df[sideband_proton_cumulative_masks["energy"]]

mc_control_evt_df = pd.concat([mc_control_evt_df, mc_control_proton_evt_df], axis=0)
'''

In [ ]:
print("Genie")
HelperFunctions.print_purity(mc_signal_df, ('truth','nu_categ','','','',''))
print("----")
HelperFunctions.print_purity(mc_control_evt_df, ('truth','nu_categ','','','',''))
print("GiBUU")
HelperFunctions.print_purity(mc_GiBUU_evt_df, ('truth','nu_categ','','','',''))

In [ ]:
HelperFunctions.print_purity(mc_signal_df[mc_signal_df.truth.nu_categ != "CC1pi"], ('truth','nu_categ','','','',''))
print("----")
HelperFunctions.print_purity(mc_control_evt_df[mc_control_evt_df.truth.nu_categ != "CC1pi"], ('truth','nu_categ','','','',''))
print("----")
HelperFunctions.print_purity(mc_GiBUU_evt_df[mc_GiBUU_evt_df.truth.nu_categ != "CC1pi"], ('truth','nu_categ','','','',''))

In [ ]:
WGT_COL = ('slc', 'wgt', '', '', '', '')

config_angle_between_candidates = FullHistogramConfig(
    file_name = "angle_between_candidates",      
    var_evt_reco_col=('slc','measure_var','angle_between_candidates','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "chi2",
    end_cut = "energy",
    bins=np.linspace(0, np.pi, 41),
    xlabel=r'Angle between MIP candidates [rad]',
    cut_value = [CTE.max_angle_between_candidates],
    ylabel=slices_y_label
)

config_max_angle_between_candidates = FullHistogramConfig(
    file_name = "angle_between_candidates",      
    var_evt_reco_col=('slc','measure_var','max_angle_between_candidates','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "chi2",
    end_cut = "energy",
    bins=np.linspace(0, np.pi, 41),
    xlabel=r'Angle between MIP candidates [rad]',
    cut_value = [CTE.max_angle_between_candidates],
    ylabel=slices_y_label
)

config_min_angle_between_candidates = FullHistogramConfig(
    file_name = "angle_between_candidates",      
    var_evt_reco_col=('slc','measure_var','min_angle_between_candidates','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "chi2",
    end_cut = "energy",
    bins=np.linspace(0, np.pi, 41),
    xlabel=r'Angle between MIP candidates [rad]',
    cut_value = [CTE.max_angle_between_candidates],
    ylabel=slices_y_label
)

config_analysis_primary_len = FullHistogramConfig(
    file_name = "analysis_primary_len", 
    var_evt_reco_col=('pfp','trk','len','','',''),
    truth_column=nu_categ_column,
    first_per_slice = True,
    start_cut = "nu_score",
    end_cut = "track",
    extra_mask = "analysis_primary",
    bins=np.linspace(0, 100, 41),
    xlabel='primary pfp lenght [cm] (vtx dist inc.)',
    ylabel= pfps_y_label,
    cut_value = [CTE.min_track_lenght],
    clip = False
)

config = config_angle_between_candidates
fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_signal_df, data_df=mc_signal_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

config = config_max_angle_between_candidates
fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_GiBUU_evt_df, data_df=mc_GiBUU_evt_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_control_evt_df, data_df=mc_control_evt_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

config = config_min_angle_between_candidates
fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_control_evt_df, data_df=mc_control_evt_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

config_vec = final_var_configs

'''
for config in config_vec:
    fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_signal_df, data_df=mc_signal_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

    fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_GiBUU_evt_df, data_df=mc_GiBUU_evt_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )
'''


'''
len_col = ('pfp', 'trk', 'len', '', '', '')
idx_max = mc_control_evt_df[is_MIP_candidate_mask(mc_control_evt_df)].groupby(level=[0, 1])[[len_col]].idxmin()

# Note: .idxmin() on a DataFrame returns a Series where the index is 
# the group and the value is the MultiIndex of the row. 
# We need just the values (the row indices) for the mask.
idx_max_values = idx_max[len_col]

longest_particle_mask = mc_control_evt_df.index.isin(idx_max_values)
longest_particles_df = mc_control_evt_df[is_MIP_candidate_mask(mc_control_evt_df) & longest_particle_mask]
config = config_analysis_primary_len
fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=longest_particles_df, data_df=longest_particles_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )


fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=longest_particles_df, data_df=longest_particles_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )




config_cos_theta_pi = FullHistogramConfig(
    file_name = "chi2_mu_mu",      
    var_evt_reco_col=('slc','measure_var','reco_cos_theta_pi','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-1, 1, 41),
    xlabel=r'Pion candidate $cos_{\theta_z}$',
    ylabel=slices_y_label
)

config_cos_theta_mu = FullHistogramConfig(
    file_name = "cos_theta_mu",      
    var_evt_reco_col=('slc','measure_var','reco_cos_theta_mu','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-1, 1, 41),
    xlabel=r'Muon candidate $cos_{\theta_z}$',
    ylabel=slices_y_label,
    stats_horizontal_alignment = 'left'
)

config_p_pi = FullHistogramConfig(
    file_name = "pion_p",      
    var_evt_reco_col=('slc','measure_var','TLE_p_pi','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins= np.array([0.13, 0.205, 0.28,0.35, 0.45,0.8]),
    xlabel=r'Pion candidate P [GeV]',
    ylabel=slices_y_label
)

fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_signal_df, data_df=mc_signal_df, config=config_cos_theta_mu,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_signal_df, data_df=mc_signal_df, config=config_cos_theta_pi,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_signal_df, data_df=mc_signal_df, config=config_p_pi,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

# Sideband


fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_control_evt_df, data_df=mc_control_evt_df, config=config_cos_theta_mu,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )


fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=mc_control_evt_df, data_df=mc_control_evt_df, config=config_cos_theta_pi,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )

# 1. Define columns
len_col = ('pfp', 'trk', 'len', '', '', '')
new_col = ('analysis', 'vars', 'len_diff_short', '', '', '')

# 2. Get sorted tracks for MIP candidates
# We group by your specific levels to ensure we are within one slice
mask = is_MIP_candidate_mask(mc_control_evt_df)
sorted_df = mc_control_evt_df[mask].sort_values(by=len_col)

# 3. Get shortest and second shortest 
# Using a list for the levels to match your index names
levels = ['__ntuple', 'entry', 'rec.slc..index']
shortest = sorted_df.groupby(level=levels)[[len_col]].nth(0)[len_col]
second_shortest = sorted_df.groupby(level=levels)[[len_col]].nth(1)[len_col]

# 4. Calculate the difference Series
# This Series will now be indexed by (__ntuple, entry, rec.slc..index)
diff_series = second_shortest - shortest

# 5. Convert to a dictionary for a safe, index-agnostic mapping
diff_dict = diff_series.to_dict()

# 6. Map back to the main dataframe
# We iterate through the index levels to build the lookup key
mc_control_evt_df[new_col] = [
    diff_dict.get((ntup, ent, slc)) 
    for ntup, ent, slc, _ in mc_control_evt_df.index
]


config_analysis_len_diff_short = FullHistogramConfig(
    file_name = "analysis_primary_len", 
    var_evt_reco_col=('pfp','vars','len_diff_short','','',''),
    truth_column=nu_categ_column,
    first_per_slice = True,
    start_cut = "nu_score",
    end_cut = "track",
    extra_mask = "analysis_primary",
    bins=np.linspace(0, 100, 41),
    xlabel='primary pfp lenght [cm] (vtx dist inc.)',
    ylabel= pfps_y_label,
    cut_value = [CTE.min_track_lenght],
    clip = False
)

config = config_analysis_len_diff_short

fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=longest_particles_df, data_df=longest_particles_df, config=config,
                cov_frac_matrix=None, cov_matrix=None,
                title="", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=False, divide_by_bin_width = False
            )
'''

# Final vars

In [ ]:

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

In [ ]:
for var_config in var_configs:       
    fig, chi2 = plot_stacked_histogram_with_ratio(
        mc_GiBUU_evt_df[mc_GiBUU_evt_df.truth.nu_categ != "CC1pi"],
        mc_GiBUU_evt_df[mc_GiBUU_evt_df.truth.nu_categ != "CC1pi"],
        var_config,
        cov_frac_matrix = None,
        cov_matrix = None,
        weight_column = ('slc', 'wgt','','','',''),
        data_pot = data_tot_pot,
        show_stats = False,
        symmetric_ratio =  True,
        divide_by_bin_width = True
        )
    
    fig, chi2 = plot_stacked_histogram_with_ratio(
            mc_signal_df[mc_signal_df.truth.nu_categ != "CC1pi"],
            mc_signal_df[mc_signal_df.truth.nu_categ != "CC1pi"],
            var_config,
            cov_frac_matrix = None,
            cov_matrix = None,
            weight_column = ('slc', 'wgt','','','',''),
            data_pot = data_tot_pot,
            show_stats = False,
            symmetric_ratio =  True,
            divide_by_bin_width = True
            )

    fig, chi2 = plot_stacked_histogram_with_ratio(
            mc_control_evt_df[mc_control_evt_df.truth.nu_categ != "CC1pi"],
            mc_control_evt_df[mc_control_evt_df.truth.nu_categ != "CC1pi"],
            var_config,
            cov_frac_matrix = None,
            cov_matrix = None,
            weight_column = ('slc', 'wgt','','','',''),
            data_pot = data_tot_pot,
            show_stats = False,
            symmetric_ratio =  True,
            divide_by_bin_width = True
            )

    
    

In [ ]:
# --- CONFIGURATION TOGGLES ---
HIDE_COSMIC_BAR = False  
USE_LOG_SCALE = False
X_MIN_LOG = 10000     
# -----------------------------
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_selection_summary(masks_dict, df_source, title_suffix="", save_filename=None):
    """
    Processes masks and generates side-by-side Efficiency and Purity plots.
    """
    plot_data = []
    
    # 1. Process Masks
    for stage_name, mask in masks_dict.items():
        # Apply mask and group
        temp_df = df_source[mask].groupby(['__ntuple', 'entry', 'rec.slc..index']).first()
        
        if HIDE_COSMIC_BAR and stage_name == "cosmic":
            continue
            
        # Group by category and sum weights
        filtered_categs = temp_df.loc[:, [('truth','nu_categ','','','',''), pot_weight_col]]
        counts = filtered_categs.groupby([('truth','nu_categ','','','','')])[[pot_weight_col]].sum()
        
        counts = counts.iloc[:, 0]
        counts.name = stage_name
        plot_data.append(counts)

    # 2. Prepare DataFrames
    df_results = pd.concat(plot_data, axis=1).T.fillna(0)

    # Column ordering for CC1pi
    signal_tag = 'CC1pi'
    if signal_tag in df_results.columns:
        cols = [signal_tag] + [c for c in df_results.columns if c != signal_tag]
        df_results = df_results[cols]

    # Map colors and rename
    current_colors = [category_colors.get(col, "#000000") for col in df_results.columns]
    df_results_renamed = df_results.rename(columns=bkg_name_nice_map).rename(index=cut_name_nice_map)

    # 3. Create Subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

    # --- Efficiency Plot (Left) ---
    df_plot_efficiency = df_results_renamed[df_results_renamed.sum(axis=1) > 0]
    df_plot_efficiency.plot(kind='barh', stacked=True, ax=ax1, color=current_colors, legend=True)

    if USE_LOG_SCALE:
        ax1.set_xscale('log')
        max_val = df_plot_efficiency.sum(axis=1).max() * 5 
        ax1.set_xlim(X_MIN_LOG, max_val)

    ax1.set_title(f"$\\nu_\\mu$CC1$\\pi$ {title_suffix} Efficiency", fontsize=20, pad=15)
    ax1.set_xlabel("Candidate Slices", fontsize=18)
    ax1.grid(True, which="both", ls="-", alpha=0.2)
    ax1.tick_params(axis='both', labelsize=16)
    ax1.legend(loc='upper right', fontsize=18, framealpha=1.0, edgecolor='black', fancybox=False)

    # --- Purity Plot (Right) ---
    df_purity = df_results_renamed.div(df_results_renamed.sum(axis=1), axis=0)
    df_purity.plot(kind='barh', stacked=True, ax=ax2, color=current_colors, legend=False)

    ax2.set_title(f"$\\nu_\\mu$CC1$\\pi$ {title_suffix} Purity", fontsize=20, pad=15)
    ax2.set_xlabel("Fraction of Total Slices", fontsize=18)
    ax2.set_xlim(0, 1)
    ax2.set_xticks(np.arange(0, 1.1, 0.1))
    ax2.grid(True, axis='x', ls='--', alpha=0.6, color='gray', zorder=0)
    ax2.tick_params(axis='both', labelsize=16)

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.08)

    # 4. Save Logic
    if save_filename:
        parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/PurEffGraphs"
        if not os.path.exists(parent_path):
            os.makedirs(parent_path)
        fig.savefig(os.path.join(parent_path, save_filename), format='pdf', bbox_inches='tight')
    
    plt.show()

# --- EXECUTION ---

# 1. Run for standard selection
plot_selection_summary(
    cumulative_masks, 
    mc_evt_df, 
    title_suffix="Selection", 
    save_filename="cc1pi_selection_summary.pdf"
)

# 2. Run for sideband
plot_selection_summary(
    sideband_cumulative_masks, 
    mc_evt_df, 
    title_suffix="Sideband", 
    save_filename="cc1pi_sideband_summary.pdf"
)

In [ ]:
signal_col = '$\\nu_{\\mu}$CC1$\\pi^{\\pm}$'
total_per_stage = df_results_renamed.sum(axis=1)

# 2. Calculate Purity: Signal / Total at each stage
purity = (df_results_renamed[signal_col] / total_per_stage) * 100

# 3. Calculate Efficiency: Signal at stage / Signal at first stage
# (Recreating num_events_0 logic)
initial_signal_count = df_results_renamed[signal_col].iloc[0]
efficiency = (df_results_renamed[signal_col] / initial_signal_count) * 100

# 4. Create a summary table for easy viewing
summary_df = pd.DataFrame({
    'Signal_Events': df_results_renamed[signal_col],
    'Total_Events': total_per_stage,
    'Purity (%)': purity,
    'Efficiency (%)': efficiency
})

print(summary_df)

In [ ]:
import numpy as np

print(f"{'Cut Name':<20} | {'Purity':<10} | {'Efficiency':<10} | {'S/sqrt(B)':<10} | {'pur * eff':<10}")
print("-" * 75)

for stage in df_results_renamed.index:
    n_signal = df_results_renamed.loc[stage, signal_col]
    n_total = total_per_stage.loc[stage]
    n_bkg = n_total - n_signal
    
    pur = (n_signal / n_total) * 100 if n_total > 0 else 0
    eff = (n_signal / initial_signal_count) * 100
    pur_eff = pur*eff/100
    s_sqrt_b = n_signal / np.sqrt(n_bkg) if n_bkg > 0 else np.nan
    
    print(f"{stage:<20} | {pur:>8.2f}% | {eff:>8.2f}% | {s_sqrt_b:>8.2f} | {pur_eff:8.2f}") 

In [ ]:
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"]], ('truth','nu_categ','','','',''))

In [ ]:
print("0p")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"] & mask_dict["0p"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))
print("1p")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"] & mask_dict["1p"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))
print("2p+")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"] & mask_dict["2plusp"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))